# `mcmlnet` Inference: Quick Start

In [ ]:
import copy
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch

from mcmlnet.training.models.base_model import BaseModel
from mcmlnet.utils.convenience import (
    load_trained_model,
    predict_in_batches,
    prepare_surrogate_model_data,
    run_model_from_physical_data,
)

%matplotlib inline
%load_ext autoreload
%autoreload 2

## Simple Inference

In the simplest use case, define a 2D NumPy array of optical parameters—absorption $\mu_a$, scattering $\mu_s$, anisotropy $g$, and refractive index $n$ (the minimum required set) and pass it to `run_model_from_physical_data`.

A 2D array with one entry per row produces a fully homogeneous, semi‑infinite tissue model; internally, the expected three‑layer structure is filled automatically.

To specify layered tissue, provide two or three entries per row to define the lower layers.

<img src="assets/tissue_model.png" width="150" class="center" alt="Three Layer Tissue Model">

In [ ]:
# Flat absorption, linear scattering, anisotropy and refractive index
n_samples = 10_000
mu_a = np.ones((n_samples, 1))  # 1/m
mu_s = np.linspace(400, 450000, n_samples).reshape(-1, 1)  # 1/m
g = np.linspace(0.8, 0.95, n_samples).reshape(-1, 1)
n = np.linspace(1.33, 1.54, n_samples).reshape(-1, 1)

# Run inference (loads pre-selected model under the hood)
reflectance_values = run_model_from_physical_data(mu_a, mu_s, g, n, d=None)

# Plot results
plt.figure(figsize=(10, 4))
plt.plot(reflectance_values)
plt.xlabel("Wavelengths [a.u.]")
plt.ylabel("Reflectance")
plt.title("Surrogate Model Reflectance Predictions")
plt.grid()
plt.show()

In [ ]:
# Multilayer use
n_samples = 10_000

mu_a1 = np.ones((n_samples, 1)) * 1000  # 1/m
mu_a2 = np.linspace(1, 1000, n_samples).reshape(-1, 1)  # 1/m
mu_a3 = np.linspace(1000, 1, n_samples).reshape(-1, 1)  # 1/m
mu_a = np.concatenate((mu_a1, mu_a2, mu_a3), axis=1)

mu_s = np.ones((n_samples, 3)) * 100_000  # 1/m
g = np.linspace(0.8, 0.95, n_samples).reshape(-1, 1).repeat(3, 1)
n = np.linspace(1.33, 1.54, n_samples).reshape(-1, 1).repeat(3, 1)
d = np.ones((n_samples, 3)) * 0.0005

print(f"Shapes: {mu_a.shape=}, {mu_s.shape=}, {g.shape=}, {n.shape=}, {d.shape=}")

# Run inference (loads pre-selected model under the hood)
reflectance_values = run_model_from_physical_data(mu_a, mu_s, g, n, d=None)

# Plot results
plt.figure(figsize=(10, 4))
plt.plot(reflectance_values)
plt.xlabel("Wavelengths [a.u.]")
plt.ylabel("Reflectance")
plt.title("Surrogate Model Reflectance Predictions")
plt.grid()
plt.show()

If any input falls outside the surrogate’s training range, a warning is issued:

In [ ]:
n_samples = 10_000
mu_a = np.linspace(0.19, 80001, n_samples).reshape(-1, 1)  # 1/m
mu_s = np.linspace(399, 450001, n_samples).reshape(-1, 1)  # 1/m
g = np.linspace(0.79, 0.96, n_samples).reshape(-1, 1)
n = np.linspace(1.32, 1.55, n_samples).reshape(-1, 1)

# Run inference (loads pre-selected model under the hood)
reflectance_values = run_model_from_physical_data(mu_a, mu_s, g, n, d=None)

# Plot results
plt.figure(figsize=(10, 4))
plt.plot(reflectance_values)
plt.xlabel("Wavelengths [a.u.]")
plt.ylabel("Reflectance")
plt.title("Surrogate Model Reflectance Predictions")
plt.grid()
plt.show()

## Custom Inference

By default, the notebook loads the standard MLP surrogate on every `run_model_from_physical_data` function call.

If you are running multiple experiments or switching between models, you can reduce loading overhead by explicitly loading the model you want once and reusing it across calls. Loading a model checkpoint of your choice is the first step into creating a custom inference pipeline, which can also allow differentiable simulation and passing of gradients.

### Load Default MLP Surrogate Model

In [ ]:
base_path = os.path.join(
    os.environ["data_dir"], "models/MLP_100M_photons_fold_0_tdr_6.0/"
)
checkpoint_path = "checkpoints/ForwardSurrogateModel-epoch=999-val_loss=0.0000.ckpt"

mlp_model, mlp_preprocessor, mlp_cfg = load_trained_model(
    base_path, checkpoint_path, BaseModel
)

### Define and Preprocess Optical Parameter Data

Define your inputs as either physiological or physical parameters - the model preprocessing supports both. Make sure you follow the expected format:
- For physiological data, the format is defined by `PhysiologicalToPhysicalTransformer.transform_hb_format`: an array of shape `(n_samples, n_layers * 8)` with per-layer parameters ordered as `[sao2, vhb, a_mie, b_mie, a_ray, g, n, d]`

NOTE: The preprocessor handles parameters and reflectances (during training) simulataneously, we therefore need to also pass dummy reflectances (in the current implementation, could - and probably should - change in future).

In [ ]:
n_samples = 100
physiological_parameters = torch.cat(
    [
        torch.linspace(0, 1, n_samples)[:, None],  # sto2
        torch.ones((n_samples, 1)) * 0.05,  # vhb
        torch.ones((n_samples, 1)) * 10000,  # a_mie [1/m]
        torch.ones((n_samples, 1)) * 2,  # b_mie
        torch.zeros((n_samples, 1)),  # a_ray
        torch.ones((n_samples, 1)) * 0.85,  # g
        torch.ones((n_samples, 1)) * 1.35,  # n
        torch.ones((n_samples, 1)) * 0.001,  # d [m]
    ],
    dim=1,
).repeat(1, 3)

# Add dummy reflectances
print("n_wavelengths used during default training:", mlp_preprocessor.n_wavelengths)
physiological_parameters = torch.cat(
    [
        physiological_parameters,
        torch.zeros(
            (physiological_parameters.shape[0], mlp_preprocessor.n_wavelengths)
        ),
    ],
    dim=-1,
)
preprocessed_physiological_data = (
    prepare_surrogate_model_data(
        mlp_preprocessor,
        mlp_cfg,
        physiological_parameters,
    )
    .float()
    .cuda()
)

# Remove dummy reflectances
preprocessed_physiological_data = preprocessed_physiological_data[..., :-1]
preprocessed_physiological_data.shape

- Physical parameters are expected to be of shape `(n_samples, n_wavelengths, n_layers * 5)`, with the last (parameter) axis ordered as `[mu_a,1, mu_a,2, mu_a,3, mu_s,1, mu_s,2, mu_s,3, g_1, g_2, g_3, n_1, n_2, n_3, d_1, d_2, d_3]`.

In [ ]:
n_samples = 10_000
physical_parameters = torch.cat(
    [
        torch.ones((1, n_samples, 1)),  # 1/m
        torch.linspace(400, 450000, n_samples).reshape(1, -1, 1),  # 1/m
        torch.linspace(0.8, 0.95, n_samples).reshape(1, -1, 1),
        torch.linspace(1.33, 1.54, n_samples).reshape(1, -1, 1),
        torch.ones((1, n_samples, 1)) * 0.001,  # m
    ],
    dim=-1,
).repeat_interleave(dim=-1, repeats=3)

# To change the expected number of wavelengths, manipulate the preprocessor's properties
# NOTE: A different physiological wavelength resolution can similarly be achieved by
#       manipulating the properties of PhysiologicalToPhysicalTransformer, including the
#       extinction coefficients, however, it is easier to implement a custom physical
#       parameters calculation based on the physiological quantities of interest and
#       pass them as 3D tensor for input normalization only.
manip_preprocessor = copy.deepcopy(mlp_preprocessor)
manip_preprocessor.n_wavelengths = physical_parameters.shape[1]

# Add dummy reflectance
physical_parameters = torch.cat(
    [
        physical_parameters,
        torch.zeros((physical_parameters.shape[0], physical_parameters.shape[1], 1)),
    ],
    dim=-1,
)
preprocessed_physical_data = (
    prepare_surrogate_model_data(
        mlp_preprocessor,
        mlp_cfg,
        physical_parameters,
    )
    .float()
    .cuda()
)

# Remove dummy reflectances
preprocessed_physical_data = preprocessed_physical_data[..., :-1]
preprocessed_physical_data.shape

### Run Inference

You can run inference in two ways:

- `model.forward` for standard, in‑memory batches, or
- `predict_in_batches` for iterating over large datasets that don’t fit in VRAM.

`predict_in_batches` also supports a progress bar disabling gradients via the `progress_bar` and `requires_grad` arguments.

In [ ]:
# Use default model forward pass
reflectances_physiological = mlp_model(preprocessed_physiological_data)
# Use predict_in_batches for large dataset that do not fully fit into VRAM
reflectances_physiological_batch = predict_in_batches(
    mlp_model,
    preprocessed_physiological_data,
    batch_size=10,
    progress_bar=False,
    requires_grad=False,
)
print(reflectances_physiological.shape, reflectances_physiological_batch.shape)

# Use default model forward pass
reflectances_physical = mlp_model(preprocessed_physical_data)
# Use predict_in_batches for large dataset that do not fully fit into VRAM
reflectances_physical_batch = predict_in_batches(
    mlp_model,
    preprocessed_physical_data,
    batch_size=10,
    progress_bar=False,
    requires_grad=False,
)
print(reflectances_physical.shape, reflectances_physical_batch.shape)

#### Plot Physiological Tissue Model Results

In [ ]:
# Create wavelength grid of the 15 training wavelengths
two_nm_grid_wvls = np.linspace(300, 1000, 351)
reduction_factor = len(two_nm_grid_wvls) // 15
wavelengths = two_nm_grid_wvls[: 15 * reduction_factor : reduction_factor]

In [ ]:
# Create underlying sto2 colormap ([0, 100]%)
cmap = plt.cm.RdBu
norm = mpl.colors.Normalize(vmin=0, vmax=100)
colors = cmap(norm(torch.linspace(0, 100, len(reflectances_physiological))))

# Plot results
fig, ax = plt.subplots(figsize=(10, 4))
for i, sample in enumerate(reflectances_physiological):
    ax.plot(wavelengths, sample.cpu().numpy(), color=colors[i])
    ax.plot(
        wavelengths,
        reflectances_physiological_batch.numpy()[i, :],
        color=colors[i],
        linestyle="--",
    )

ax.set_xlabel("Wavelengths [nm]")
ax.set_ylabel("Reflectance")
ax.set_title("Surrogate Model Reflectance Predictions using Hemoglobin as Absorber")
ax.grid(True)

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label(r"s$_t$O$_2$ [%]")

plt.show()

#### Plot Physical Tissue Model Results

In [ ]:
# Plot results
plt.figure(figsize=(10, 4))
plt.plot(reflectances_physical.cpu().numpy().squeeze(), label="model.forward")
plt.plot(reflectances_physical_batch.numpy().squeeze(), linestyle="--", label="batched")
plt.xlabel("Wavelengths [a.u.]")
plt.ylabel("Reflectance")
plt.title("Surrogate Model Reflectance Predictions")
plt.grid()
plt.legend()
plt.show()

The last two plots showcased
1. the physiological inference capabilities for different tissue oxygenation levels and
2. reused the parameters of the initial, "simple inference" example to walk you through the individual, sometimes slightly odd, pipeline steps.

This concludes the quick‑start example for surrogate inference. From here, you can swap in your own optical parameters, adjust preprocessing, run larger batches, or re-assemble and re-write source code as needed.